# 03 — Prebuilt Invoice: Process Repair & Medical Invoices

**Time**: ~10 minutes · **Model**: `prebuilt-invoice` · **No training required**

---

## Insurance Scenario

A policyholder submits a **repair invoice** (auto insurance) or **medical invoice** (health insurance) as part of a claim. The claims team needs to extract:

- Vendor name and address
- Invoice number and date
- Line items with descriptions, quantities, and amounts
- Total amount, tax, and due date

The **Invoice model** extracts all of these fields automatically.

---

## Prerequisites

Complete [00-setup-and-basics.ipynb](00-setup-and-basics.ipynb) first.

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, AnalyzeResult
import pandas as pd

load_dotenv(dotenv_path=os.path.join("..", ".env"))

_api_key = os.environ.get("DOCUMENTINTELLIGENCE_API_KEY", "")
_credential = AzureKeyCredential(_api_key) if _api_key else DefaultAzureCredential()

client = DocumentIntelligenceClient(
    endpoint=os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"],
    credential=_credential
)
print("✅ Client ready")

## Analyze an Invoice

In [ ]:
# Sample invoice (replace with a real repair/medical invoice)
invoice_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-invoice.pdf"

poller = client.begin_analyze_document(
    "prebuilt-invoice",
    AnalyzeDocumentRequest(url_source=invoice_url)
)
result: AnalyzeResult = poller.result()

print(f"Invoices found: {len(result.documents) if result.documents else 0}")

## Extract Invoice Header Fields

These are the key fields your claims system needs for automated processing.

In [ ]:
def get_field(fields, name):
    """Safely extract a field value and confidence."""
    field = fields.get(name)
    if field:
        return field.get("content", "N/A"), field.get("confidence", 0)
    return "Not found", 0.0

if result.documents:
    invoice = result.documents[0]
    fields = invoice.fields
    
    print("=" * 60)
    print("INVOICE SUMMARY")
    print("=" * 60)
    
    # Key header fields for insurance claim processing
    header_fields = [
        "VendorName", "VendorAddress", 
        "CustomerName", "CustomerAddress",
        "InvoiceId", "InvoiceDate", "DueDate",
        "SubTotal", "TotalTax", "InvoiceTotal", "AmountDue"
    ]
    
    for field_name in header_fields:
        value, confidence = get_field(fields, field_name)
        flag = "⚠️" if confidence < 0.80 and confidence > 0 else "✅" if confidence > 0 else "  "
        print(f"  {flag} {field_name:30s} {str(value):30s} ({confidence:.0%})")

## Extract Line Items

**Why this matters for insurance**: Line items show exactly what's being charged — repair parts, labor hours, medical procedures. This is essential for claim validation and fraud detection.

In [ ]:
if result.documents:
    invoice = result.documents[0]
    items = invoice.fields.get("Items")
    
    if items and items.get("valueArray"):
        line_items = []
        for idx, item in enumerate(items["valueArray"]):
            obj = item.get("valueObject", {})
            line_items.append({
                "#": idx + 1,
                "Description": obj.get("Description", {}).get("content", "N/A")[:40],
                "Quantity": obj.get("Quantity", {}).get("content", "N/A"),
                "Unit Price": obj.get("UnitPrice", {}).get("content", "N/A"),
                "Amount": obj.get("Amount", {}).get("content", "N/A"),
                "Tax": obj.get("Tax", {}).get("content", "N/A"),
            })
        
        df = pd.DataFrame(line_items)
        print("\nLINE ITEMS:")
        print(df.to_string(index=False))
    else:
        print("No line items found.")

## Confidence Overview — All Fields

For an insurance workflow, you want a quick overview of which extracted fields are reliable and which need human review.

In [ ]:
if result.documents:
    invoice = result.documents[0]
    confidence_data = []
    
    for name, field in invoice.fields.items():
        if name == "Items":
            continue  # Skip array field
        conf = field.get("confidence", 0)
        confidence_data.append({
            "Field": name,
            "Confidence": conf,
            "Status": "✅ Auto-accept" if conf >= 0.90 else "⚠️ Review" if conf >= 0.70 else "❌ Manual entry"
        })
    
    df = pd.DataFrame(confidence_data).sort_values("Confidence", ascending=True)
    print("\nFIELD CONFIDENCE OVERVIEW:")
    print(df.to_string(index=False))
    
    # Summary
    auto = sum(1 for c in confidence_data if c["Confidence"] >= 0.90)
    review = sum(1 for c in confidence_data if 0.70 <= c["Confidence"] < 0.90)
    manual = sum(1 for c in confidence_data if c["Confidence"] < 0.70)
    print(f"\n📊 Summary: {auto} auto-accept, {review} needs review, {manual} manual entry")

## Build a Claim-Ready Data Structure

Transform the extracted data into a structure your claims system can consume.

In [ ]:
import json

if result.documents:
    invoice = result.documents[0]
    f = invoice.fields
    
    claim_record = {
        "vendor": {
            "name": f.get("VendorName", {}).get("content"),
            "address": f.get("VendorAddress", {}).get("content"),
        },
        "invoice": {
            "id": f.get("InvoiceId", {}).get("content"),
            "date": f.get("InvoiceDate", {}).get("content"),
            "due_date": f.get("DueDate", {}).get("content"),
            "total": f.get("InvoiceTotal", {}).get("content"),
            "tax": f.get("TotalTax", {}).get("content"),
        },
        "needs_review": any(
            f.get(key, {}).get("confidence", 0) < 0.80
            for key in ["InvoiceTotal", "VendorName", "InvoiceDate"]
        ),
    }
    
    print("Claim-ready JSON:")
    print(json.dumps(claim_record, indent=2))

---

## Key Takeaways

| Feature | Value for Insurance |
|---|---|
| **Vendor extraction** | Identify repair shops, medical providers, service vendors |
| **Line item parsing** | Validate individual charges against policy coverage |
| **Amount extraction** | Auto-populate claim reimbursement amounts |
| **Confidence scoring** | Route uncertain invoices to adjusters for manual review |

## Supported Invoice Fields

The `prebuilt-invoice` model extracts 20+ fields. Full list: [Invoice model documentation](https://learn.microsoft.com/en-us/azure/ai-services/document-intelligence/prebuilt/invoice?view=doc-intel-4.0.0#field-extraction)

## Next Steps

| Notebook | What You'll Learn |
|---|---|
| [04-prebuilt-receipt.ipynb](04-prebuilt-receipt.ipynb) | Process expense receipts for claims |
| [10-human-in-the-loop.ipynb](10-human-in-the-loop.ipynb) | Build a confidence-based review workflow |